### Needed for Login

In [ ]:
# !pip install earthscope-cli
# !pip install boto3
# !pip install earthscope-sdk
# !pip install obspy

In [ ]:
!curl ipinfo.io

{
  "ip": "34.125.182.151",
  "hostname": "151.182.125.34.bc.googleusercontent.com",
  "city": "Las Vegas",
  "region": "Nevada",
  "country": "US",
  "loc": "36.1750,-115.1372",
  "org": "AS396982 Google LLC",
  "postal": "89111",
  "timezone": "America/Los_Angeles",
  "readme": "https://ipinfo.io/missingauth"
}

In [ ]:
!es login

Attempting to automatically open the SSO authorization page in your default 
browser.
                        If the browser does not open or you wish to use a 
different device to authorize this request, open the following URL:

                        https://login.earthscope.org/activate?user_code=DLBJ-PMM
J
Successful login! Access token expires at 2026-03-14 23:41:47+00:00


In [ ]:
import boto3
from botocore.config import Config
from earthscope_sdk import EarthScopeClient
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
client = EarthScopeClient()

profile = client.user.get_profile()
print(profile)

first_name='Tolulope' last_name='Olugboji' country_code='US' region_code='NY' institution='University of Rochester' work_sector='Education' user_id='auth0|68f1e0de88431a7cc5fc1a08' primary_email='tolulope.olugboji@rochester.edu' created_at=datetime.datetime(2025, 10, 17, 6, 24, 24, 418355, tzinfo=TzInfo(0)) updated_at=datetime.datetime(2025, 10, 17, 6, 24, 24, 418363, tzinfo=TzInfo(0))


In [ ]:
creds = client.user.get_aws_credentials()
print("Credentials retrieved successfully.")

Credentials retrieved successfully.


In [ ]:
# Initialize S3 Client
session = boto3.Session(
    aws_access_key_id=creds.aws_access_key_id,
    aws_secret_access_key=creds.aws_secret_access_key,
    aws_session_token=creds.aws_session_token,
)

s3_client = session.client("s3",config=Config(response_checksum_validation='when_required'))

In [ ]:
S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT
PREFIX = "miniseed/"

### Check What Networks are available

In [ ]:
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
nets = [c["Prefix"].split("/", 1)[1] for c in list_resp["CommonPrefixes"]]
print(nets)

['12/', '16/', '17/', '1A/', '1B/', '1C/', '1D/', '1E/', '1F/', '1G/', '1H/', '1J/', '1K/', '1L/', '1M/', '1P/', '1Q/', '1R/', '1T/', '1U/', '1V/', '1Z/', '22/', '24/', '28/', '29/', '2A/', '2B/', '2C/', '2D/', '2E/', '2F/', '2G/', '2H/', '2I/', '2J/', '2K/', '2L/', '2M/', '2O/', '2P/', '2Q/', '2U/', '2V/', '34/', '3A/', '3B/', '3C/', '3D/', '3E/', '3F/', '3H/', '3J/', '3K/', '3L/', '3R/', '3U/', '3W/', '3Y/', '4A/', '4B/', '4E/', '4F/', '4H/', '4I/', '4J/', '4N/', '4O/', '4P/', '4Q/', '4R/', '4S/', '4T/', '4U/', '4Y/', '4Z/', '5A/', '5B/', '5C/', '5E/', '5F/', '5G/', '5H/', '5I/', '5J/', '5K/', '5L/', '5O/', '5P/', '5Q/', '5S/', '5W/', '6A/', '6B/', '6C/', '6D/', '6E/', '6F/', '6G/', '6H/', '6I/', '6J/', '6K/', '6L/', '6M/', '6N/', '6O/', '6Q/', '6R/', '6W/', '7A/', '7B/', '7C/', '7D/', '7E/', '7F/', '7G/', '7I/', '7J/', '7K/', '7L/', '7O/', '7P/', '7Q/', '7S/', '7T/', '7U/', '8A/', '8B/', '8E/', '8F/', '8G/', '8H/', '8I/', '8J/', '8L/', '8P/', '8Q/', '8S/', '8U/', '8W/', '9A/', '9B/'

In [ ]:
get_resp = s3_client.get_object(
    Bucket=BUCKET,
    #Key=f"{PREFIX}IU/2024/300/MBW.UW.2024.300#2",  # ~6 MB
    # Key=f"{PREFIX}UW/2024/300/MPO.UW.2024.300#2",  # ~85 MB
    Key=f"{PREFIX}UW/2024/300/SLA.UW.2024.300#2",  # ~400 MB
)

ClientError: An error occurred (Forbidden) when calling the GetObject operation: Forbidden.

In [ ]:
print(dir(s3_client))

['_PY_TO_OP_NAME', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_cache', '_client_config', '_convert_to_request_dict', '_emit_api_params', '_endpoint', '_exceptions', '_exceptions_factory', '_get_credentials', '_get_waiter_config', '_load_exceptions', '_loader', '_make_api_call', '_make_request', '_register_handlers', '_request_signer', '_resolve_endpoint_ruleset', '_response_parser', '_ruleset_resolver', '_serializer', '_service_model', '_user_agent_creator', 'abort_multipart_upload', 'can_paginate', 'close', 'complete_multipart_upload', 'copy', 'copy_object', 'create_bucket', 'create_bucket_metadata_configuration', 'create_bucket_metadata_table_configuration'

### List of All Data Available for a Particular Network

In [ ]:
## Example to browse through the cloud
network = 'IM'
network_prefix = f"miniseed/{network}/2024/065/"
list_resp_years = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=network_prefix)

years = []
years = [c["Key"].split('/') for c in list_resp_years["Contents"]]

print(years)
len(list_resp_years["Contents"])

[['miniseed', 'IM', '2024', '065', 'BC01.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BC02.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BC03.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BC04.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BC05.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BM03.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BM04.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'BM05.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY101.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY201.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY302.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY303.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY306.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY501.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY601.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY602.IM.2024.065#1'], ['miniseed', 'IM', '2024', '065', 'CY603.IM.2024.065#1'], ['miniseed', 'IM', '2

200

In [ ]:
print(nets)

['12/', '16/', '17/', '1A/', '1B/', '1C/', '1D/', '1E/', '1F/', '1G/', '1H/', '1J/', '1K/', '1L/', '1M/', '1P/', '1Q/', '1R/', '1T/', '1U/', '1V/', '1Z/', '22/', '24/', '28/', '29/', '2A/', '2B/', '2C/', '2D/', '2E/', '2F/', '2G/', '2H/', '2I/', '2J/', '2K/', '2L/', '2M/', '2O/', '2P/', '2Q/', '2U/', '2V/', '34/', '3A/', '3B/', '3C/', '3D/', '3E/', '3F/', '3H/', '3J/', '3K/', '3L/', '3R/', '3U/', '3W/', '3Y/', '4A/', '4B/', '4E/', '4F/', '4H/', '4I/', '4J/', '4N/', '4O/', '4P/', '4Q/', '4R/', '4S/', '4T/', '4U/', '4Y/', '4Z/', '5A/', '5B/', '5C/', '5E/', '5F/', '5G/', '5H/', '5I/', '5J/', '5K/', '5L/', '5O/', '5P/', '5Q/', '5S/', '5W/', '6A/', '6B/', '6C/', '6D/', '6E/', '6F/', '6G/', '6H/', '6I/', '6J/', '6K/', '6L/', '6M/', '6N/', '6O/', '6Q/', '6R/', '6W/', '7A/', '7B/', '7C/', '7D/', '7E/', '7F/', '7G/', '7I/', '7J/', '7K/', '7L/', '7O/', '7P/', '7Q/', '7S/', '7T/', '7U/', '8A/', '8B/', '8E/', '8F/', '8G/', '8H/', '8I/', '8J/', '8L/', '8P/', '8Q/', '8S/', '8U/', '8W/', '9A/', '9B/'

In [ ]:
## Example to Prepare a Sample Data Availability List in Cloud for a Network for Few Years
network = 'IM'
datalist = []
list_of_years = np.arange(2003,2005,1)
list_of_days = [f'{i:03d}' for i in range(1, 367)]

for years in list_of_years:
    for days in list_of_days:
      data_prefix = f"miniseed/{network}/{years}/{days}/"
      try:
        data_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=data_prefix)
        for c in data_resp["Contents"]:
            dataacess_key = c["Key"]
            year = dataacess_key.split('/')[2]
            yearday = dataacess_key.split('/')[3]
            station = dataacess_key.split('/')[-1].split('.')[0]
            datalist.append([network,station,year,yearday,dataacess_key])
      except Exception as e:
        print(e)
        continue

datalist_df = pd.DataFrame(datalist,columns=['network','station','year','yearday','dataacess_key'])
datalist_df.sort_values(by=['network','station','year','yearday'], inplace=True)
datalist_df


'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'
'Contents'


,network,station,year,yearday,dataacess_key
0,IM,ATTU,2003,001,miniseed/IM/2003/001/ATTU.IM.2003.001#1
48,IM,ATTU,2003,002,miniseed/IM/2003/002/ATTU.IM.2003.002#1
96,IM,ATTU,2003,003,miniseed/IM/2003/003/ATTU.IM.2003.003#1
144,IM,ATTU,2003,004,miniseed/IM/2003/004/ATTU.IM.2003.004#1
192,IM,ATTU,2003,005,miniseed/IM/2003/005/ATTU.IM.2003.005#1
...,...,...,...,...,...
41757,IM,VNDAB,2004,362,miniseed/IM/2004/362/VNDAB.IM.2004.362#1
41821,IM,VNDAB,2004,363,miniseed/IM/2004/363/VNDAB.IM.2004.363#1
41885,IM,VNDAB,2004,364,miniseed/IM/2004/364/VNDAB.IM.2004.364#1
41949,IM,VNDAB,2004,365,miniseed/IM/2004/365/VNDAB.IM.2004.365#1


In [ ]:
#Same as above but with all years for one station
network = '1B'
datalist = []

data_prefix = f"miniseed/{network}/"

# Create a reusable paginator
paginator = s3_client.get_paginator('list_objects_v2')

try:
    pages = paginator.paginate(Bucket=BUCKET, Prefix=data_prefix)

    for page in pages:
        if "Contents" not in page:
            continue

        for c in page["Contents"]:
            dataacess_key = c["Key"]

            parts = dataacess_key.split('/')

            if len(parts) >= 5:
                extracted_year = parts[2]
                yearday = parts[3]
                station = parts[-1].split('.')[0]

                datalist.append([network, station, extracted_year, yearday, dataacess_key])

except Exception as e:
    print(f"Error accessing prefix {data_prefix}: {e}")

datalist_df = pd.DataFrame(datalist, columns=['network', 'station', 'year', 'yearday', 'dataacess_key'])
datalist_df.sort_values(by=['network', 'station', 'year', 'yearday'], inplace=True)

if not datalist_df.empty:
    discovered_years = datalist_df['year'].unique()
    print(f"Success! Found {len(datalist_df)} files for network {network}.")
    print(f"Years found in bucket: {discovered_years}")
else:
    print(f"DataFrame is still empty. Network '{network}' does not exist in this S3 bucket.")

datalist_df

Success! Found 37289 files for network 1B.
Years found in bucket: ['2018' '2011' '2014' '2013']


,network,station,year,yearday,dataacess_key
30001,1B,1,2018,207,miniseed/1B/2018/207/1.1B.2018.207#3
30026,1B,1,2018,208,miniseed/1B/2018/208/1.1B.2018.208#2
30081,1B,1,2018,209,miniseed/1B/2018/209/1.1B.2018.209#2
30161,1B,1,2018,210,miniseed/1B/2018/210/1.1B.2018.210#2
30241,1B,1,2018,211,miniseed/1B/2018/211/1.1B.2018.211#2
...,...,...,...,...,...
110,1B,HOLE3,2013,105,miniseed/1B/2013/105/HOLE3.1B.2013.105#2
114,1B,HOLE3,2013,106,miniseed/1B/2013/106/HOLE3.1B.2013.106#2
118,1B,HOLE3,2013,107,miniseed/1B/2013/107/HOLE3.1B.2013.107#2
122,1B,HOLE3,2013,108,miniseed/1B/2013/108/HOLE3.1B.2013.108#2


In [ ]:
len(set(datalist_df['station']))

2364

In [ ]:
!ls

coordinate_matched_networks  drive  sample_data  stationcoordinates  Upload


In [ ]:
(df_coords["network"] =='1B').sum()

np.int64(2366)

In [ ]:
df_coords[df_coords["network"] == '1B']['station'].nunique()

2366

In [ ]:
# 1. Get the unique stations from your IRIS metadata
meta_stations = set(df_coords[df_coords['network'] == '1B']['station'])

# 2. Get the unique stations actually sitting in your S3 bucket
data_stations = set(datalist_df['station'])

# 3. Find stations that are in the metadata, but missing from S3
missing_from_s3 = meta_stations - data_stations

# 4. Find stations that are in S3, but missing from metadata (rare, but happens!)
missing_from_meta = data_stations - meta_stations

print(f"Stations in IRIS but missing from S3 data: {len(missing_from_s3)}")
if missing_from_s3:
    print(missing_from_s3)

print(f"\nStations in S3 data but missing from IRIS metadata: {len(missing_from_meta)}")
if missing_from_meta:
    print(missing_from_meta)

Stations in IRIS but missing from S3 data: 2
{'6R533', '6H422'}

Stations in S3 data but missing from IRIS metadata: 0


In [ ]:
datalist_df.groupby(['network','station','year']).count().reset_index()

,network,station,year,yearday,dataacess_key
0,IM,ATTU,2003,349,349
1,IM,ATTU,2004,336,336
2,IM,IL01,2003,349,349
3,IM,IL01,2004,353,353
4,IM,IL02,2003,349,349
...,...,...,...,...,...
124,IM,VNDA,2003,1,1
125,IM,VNDA1,2003,344,344
126,IM,VNDA1,2004,298,298
127,IM,VNDAB,2003,345,345


In [ ]:
datalist_df[datalist_df['station'].str.contains('ATTU')]

,network,station,year,yearday,dataacess_key
0,IM,ATTU,2003,001,miniseed/IM/2003/001/ATTU.IM.2003.001#1
48,IM,ATTU,2003,002,miniseed/IM/2003/002/ATTU.IM.2003.002#1
96,IM,ATTU,2003,003,miniseed/IM/2003/003/ATTU.IM.2003.003#1
144,IM,ATTU,2003,004,miniseed/IM/2003/004/ATTU.IM.2003.004#1
192,IM,ATTU,2003,005,miniseed/IM/2003/005/ATTU.IM.2003.005#1
...,...,...,...,...,...
41694,IM,ATTU,2004,362,miniseed/IM/2004/362/ATTU.IM.2004.362#1
41758,IM,ATTU,2004,363,miniseed/IM/2004/363/ATTU.IM.2004.363#1
41822,IM,ATTU,2004,364,miniseed/IM/2004/364/ATTU.IM.2004.364#1
41886,IM,ATTU,2004,365,miniseed/IM/2004/365/ATTU.IM.2004.365#1


### Download Data

In [ ]:
import os
from datetime import datetime

def download_daily_data(s3_client, bucket_name, net, prefix, output_dir='/content/drive/MyDrive/PRJ_Wavenet_Terraseis/sample_data'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Starting download for {prefix}")

    try:
        local_path = os.path.join(output_dir, net)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        #s3_client.download_file(bucket_name, prefix, local_path)
        s3_client.get_object(Bucket=bucket_name, Key=prefix)
        print("Received the Object")

    except Exception as e:
        print(f"Error accessing network {net}: {e}")
        print("Download process not completed.")

In [ ]:
# Configuration for 2004 Sumatra Earthquake
S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT

sumatra_date = '2004-12-26'
date_obj = datetime.strptime(sumatra_date, '%Y-%m-%d')
yearday = date_obj.strftime('%j')
year = date_obj.strftime('%Y')


network='UW'
filtered_df = datalist_df[(datalist_df['station'].str.contains('ATTU')) & (datalist_df['year']==year) & (datalist_df['yearday']==yearday)]
print(filtered_df['dataacess_key'].item())

## Run the download
#download_daily_data(s3_client, BUCKET, net='IM', prefix="miniseed/UW/2024/300/MBW.UW.2024.300#2")

s3_client.get_object(Bucket=BUCKET, Key="miniseed/UW/2024/300/MPO.UW.2024.300#2")

miniseed/IM/2004/361/ATTU.IM.2004.361#1


ClientError: An error occurred (Forbidden) when calling the GetObject operation: Forbidden.

In [ ]:
s3_client = session.client("s3")
get_resp = s3_client.get_object(
    Bucket=BUCKET,
    Key=f"{PREFIX}UW/2024/300/MBW.UW.2024.300#2",  # ~6 MB
    # Key=f"{PREFIX}UW/2024/300/MPO.UW.2024.300#2",  # ~85 MB
    # Key=f"{PREFIX}UW/2024/300/SLA.UW.2024.300#2",  # ~400 MB
)

ClientError: An error occurred (Forbidden) when calling the GetObject operation: Forbidden.

### Draft Codes

In [ ]:
# network = 'IM'
# datalist = []
# network_prefix = f"miniseed/{network}/"
# resp_years = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=network_prefix, Delimiter='/')

# if 'CommonPrefixes' in resp_years:
#     # Extract year strings from the prefixes (e.g., 'miniseed/IM/1998/' -> '1998')
#     available_years = [cp['Prefix'].split('/')[-2] for cp in resp_years['CommonPrefixes']]

#     for year in available_years:
#         # 2. For each available year, list available days
#         year_prefix = f"miniseed/{network}/{year}/"
#         resp_days = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=year_prefix, Delimiter='/')

#         if 'CommonPrefixes' in resp_days:
#             available_days = [cp['Prefix'].split('/')[-2] for cp in resp_days['CommonPrefixes']]

#             for day in available_days:
#                 # 3. List actual files in the specific day directory
#                 day_prefix = f"miniseed/{network}/{year}/{day}/"
#                 resp_files = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=day_prefix)

#                 if 'Contents' in resp_files:
#                     for c in resp_files['Contents']:
#                         dataacess_key = c['Key']
#                         # Parse filename: usually NET.STA.YEAR.JDAY...
#                         # Safely extracting parts
#                         parts = dataacess_key.split('/')
#                         # parts usually: ['miniseed', 'IM', '1998', '108', 'filename']

#                         station = parts[-1].split('.')[0]

#                         datalist.append([network, station, year, day, dataacess_key])

# datalist_df = pd.DataFrame(datalist, columns=['network', 'station', 'year', 'yearday', 'dataacess_key'])
# datalist_df.sort_values(by=['network', 'station', 'year', 'yearday'], inplace=True)

In [ ]:
# import os
# from datetime import datetime

# def download_daily_data(s3_client, bucket_name, date_str, networks, output_dir='data'):
#     """
#     Downloads data for specific networks on a specific day.

#     Args:
#     s3_client: Boto3 S3 client object.
#     bucket_name: Name of the S3 bucket.
#     date_str: Date string in 'YYYY-MM-DD' format.
#     networks: List of network codes (e.g., ['IU', 'II']).
#     output_dir: Local directory to save files.
#     """
#     target_date = datetime.strptime(date_str, '%Y-%m-%d')
#     year = target_date.strftime('%Y')
#     jday = target_date.strftime('%j')

#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)

#     print(f"Starting download for {date_str} (Year: {year}, JDay: {jday})...")

#     for net in networks:
#         # constructing a likely prefix.
#         # Note: Adjust this structure based on the actual S3 bucket layout.
#         # Common layout: miniseed/YEAR/NET/STA/...
#         prefix = f"miniseed/{year}/"

#         print(f"Scanning network: {net} with prefix: {prefix}")

#         try:
#             paginator = s3_client.get_paginator('list_objects_v2')
#             page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

#             file_count = 0
#             for page in page_iterator:
#                 if 'Contents' in page:
#                     for obj in page['Contents']:
#                         key = obj['Key']

#                         # Filter by day if the filename contains day information
#                         # Assuming filename format like: NET.STA.LOC.CHAN.D.YEAR.JDAY
#                         if f".{year}.{jday}" in key:
#                             filename = os.path.basename(key)
#                             local_path = os.path.join(output_dir, net, filename)

#                             # Create network subdirectory
#                             os.makedirs(os.path.dirname(local_path), exist_ok=True)

#                             print(f"Downloading {key} -> {local_path}")
#                             s3_client.download_file(bucket_name, key, local_path)
#                             file_count += 1

#             if file_count == 0:
#                 print(f"No files found for network {net} on {date_str} with prefix {prefix}")

#         except Exception as e:
#             print(f"Error accessing network {net}: {e}")

#     print("Download process completed.")

In [ ]:
# import boto3
# import os
# from earthscope_sdk import EarthScopeClient

# # 1. Initialize EarthScope Client and get temporary AWS Credentials
# print("Authenticating with EarthScope...")
# client = EarthScopeClient()
# creds = client.user.get_aws_credentials()

# # 2. Create a boto3 Session with the temporary credentials
# session = boto3.Session(
#     aws_access_key_id=creds.aws_access_key_id,
#     aws_secret_access_key=creds.aws_secret_access_key,
#     aws_session_token=creds.aws_session_token,
# )
# s3_client = session.client("s3")

# # 3. Define the S3 Access Point and target data
# S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
# BUCKET = S3_ACCESS_POINT

# # Define the specific day you want to download
# NETWORK = "UW"
# YEAR = "2024"
# DAY_OF_YEAR = "300" # This is the Julian day of the year

# # The prefix structure is miniseed/NETWORK/YEAR/DAY_OF_YEAR/
# target_prefix = f"miniseed/{NETWORK}/{YEAR}/{DAY_OF_YEAR}/"

# # 4. Prepare local directory for downloads
# output_dir = f"data_{NETWORK}_{YEAR}_{DAY_OF_YEAR}"
# os.makedirs(output_dir, exist_ok=True)

# # 5. List and download the data
# print(f"Searching for data in: {target_prefix}")

# # We use a paginator in case there are more than 1000 files in this day/network
# paginator = s3_client.get_paginator('list_objects_v2')
# pages = paginator.paginate(Bucket=BUCKET, Prefix=target_prefix)

# download_count = 0
# total_bytes = 0

# for page in pages:
#     if "Contents" in page:
#         for obj in page["Contents"]:
#             key = obj["Key"]
#             file_size = obj["Size"]

#             # Extract the actual filename from the end of the S3 key path
#             filename = key.split("/")[-1]
#             download_path = os.path.join(output_dir, filename)

#             print(f"Downloading {filename} ({(file_size / 1024 / 1024):.2f} MB)...")

#             # Download the file to the local directory
#             try:
#                 s3_client.download_file(BUCKET, key, download_path)
#                 download_count += 1
#                 total_bytes += file_size
#             except Exception as e:
#                 print(f"Failed to download {filename}. Error: {e}")
#                 # This might happen if you hit a restricted file you don't have access to

# print("-" * 30)
# print(f"Finished! Successfully downloaded {download_count} files.")
# print(f"Total data downloaded: {(total_bytes / 1024 / 1024):.2f} MB.")
# print(f"Files saved to: ./{output_dir}/")